In [1]:
dataset = """ 
pred1('e11', 'e21', 'e31', 'e41').
pred2('e11', 'e21', 'e32').
pred3('e12', 'e22', 'e32').
pred4('e12', 'e22').
pred5('e13', 'e21', 'e33').
pred6('e13', 'e23', 'e31', 'e42', 'e51').
pred7('e11').

pred1('e14', 'e23', 'e31', 'e41').
pred2('e11', 'e24', 'e32').
pred3('e14', 'e22', 'e32').
pred4('e15', 'e21').
pred5('e13', 'e22', 'e33').
pred6('e12', 'e24', 'e31', 'e42', 'e51').
pred7('e13').

pred1('e11', 'e21', 'e32', 'e41').
pred2('e13', 'e24', 'e32').
pred3('e12', 'e22', 'e32').
pred4('e12', 'e24').
pred5('e13', 'e21', 'e336').
pred6('e11', 'e23', 'e31', 'e43', 'e52').
pred7('e12').   

pred1('e16', 'e27', 'e38', 'e49').
pred2('e17', 'e28', 'e39').
pred3('e18', 'e29', 'e31').
pred4('e19', 'e21').
pred5('e13', 'e21', 'e31').
pred6('e11', 'e21', 'e31', 'e414', 'e52').
pred7('e13').
"""

In [2]:
dataset = """
zero(0).
succ(0, 1).
succ(1, 2).
succ(2, 3).
succ(3, 4).
succ(4, 5).
succ(5, 6).
succ(6, 7).
succ(7, 8).
succ(8, 9).
even(0).
odd(1).
even(2).
odd(3).
even(4).
odd(5).
even(6).
odd(7).
even(8).
odd(9).
"""

In [3]:
even_bk = """ 
zero(0).
successor(X1, X2) :- X2 is X1 + 1.
even(X1) :- 0 is X1 mod 2.
odd(X1) :- 1 is X1 mod 2.
"""

In [4]:
dataset = even_bk

In [5]:
from problog.program import PrologString
from problog import get_evaluatable

program = PrologString(dataset)
query = """
query(even(4)).
"""
# PrologString cannot be concatenated with a plain str, so create a new PrologString
# from the combined source (dataset is already a str)
Input = PrologString(dataset + query)
result = get_evaluatable().create_from(Input).evaluate()
print(result)

{even(4): 1.0}


In [6]:
result.keys(), result.values()

(dict_keys([even(4)]), dict_values([1.0]))

In [7]:
import re

# dataset is defined in a previous cell
pattern = re.compile(r'^\s*([A-Za-z_]\w*)\s*\(', re.MULTILINE)
predicates = sorted(set(pattern.findall(dataset)))
print(predicates)

['even', 'odd', 'successor', 'zero']


In [8]:
def _split_args(s):
    args = []
    cur = []
    q = None
    esc = False
    for ch in s:
        if esc:
            cur.append(ch); esc = False; continue
        if ch == '\\':
            esc = True; continue
        if q:
            if ch == q:
                q = None
            cur.append(ch)
            continue
        if ch in ("'", '"'):
            q = ch
            cur.append(ch)
            continue
        if ch == ',':
            args.append(''.join(cur).strip())
            cur = []
            continue
        cur.append(ch)
    last = ''.join(cur).strip()
    if last:
        args.append(last)
    return args

pred_arity_dict = {}
for pred in predicates:
    found = set()
    for m in re.finditer(r'\b' + re.escape(pred) + r'\s*\(', dataset):
        # find matching closing parenthesis (handles nested parentheses if any)
        i = m.end()  # position after '('
        depth = 1
        while i < len(dataset) and depth > 0:
            if dataset[i] == '(':
                depth += 1
            elif dataset[i] == ')':
                depth -= 1
            i += 1
        content = dataset[m.end():i-1].strip()
        if content == '':
            found.add(0)
        else:
            args = _split_args(content)
            found.add(len(args))
    pred_arity_dict[pred] = sorted(found)[0]

print(pred_arity_dict)

{'even': 1, 'odd': 1, 'successor': 2, 'zero': 1}


In [9]:
pred_arity_entity_dict = {}
for pred, arity in pred_arity_dict.items():
    pred_arity_entity_dict[pred] = {}
    for n in range(arity):
        variable_key = 'X' + str(n+1)
        pred_arity_entity_dict[pred][variable_key] = []  # Initialize empty list for each argument position

In [10]:
# iterate over each predicate occurrence in the dataset and collect entities per argument position
for pred in predicates:
    for m in re.finditer(r'\b' + re.escape(pred) + r'\s*\(', dataset):
        # find matching closing parenthesis
        i = m.end()
        depth = 1
        while i < len(dataset) and depth > 0:
            if dataset[i] == '(':
                depth += 1
            elif dataset[i] == ')':
                depth -= 1
            i += 1
        content = dataset[m.end():i-1].strip()
        if content == '':
            continue
        args = _split_args(content)
        for idx, arg in enumerate(args):
            val = arg.strip()
            # remove surrounding quotes if present
            if (val.startswith("'") and val.endswith("'")) or (val.startswith('"') and val.endswith('"')):
                val = val[1:-1]
            key = 'X' + str(idx+1)
            lst = pred_arity_entity_dict[pred][key]
            if val and val not in lst:
                lst.append(val)

# optional: show collected entities for the current predicate (pred variable available in notebook)
print(pred_arity_entity_dict[pred])

{'X1': ['0']}


In [22]:
# potential_body_preds = ['pred1(X1, X2, X3, X4)', 'pred2(X1, X2, X3)', 'pred3(X1, X2, X3)', 'pred4(X1, X2)', 'pred5(X1, X2, X3)', 'pred6(X1, X2, X3, X4, X5)', 'pred7(X1)']
potential_body_preds = ['zero(X1)', 'zero(X2)', 'zero(X3)', 'successor(X1, X2)', 'successor(X2, X3)', 'even(X1)', 'even(X2)']
head_pred = ['even(X3)']

all_preds = potential_body_preds + head_pred


In [23]:
# extract variables from potential_body_preds
vars_list = []
seen = set()
for expr in all_preds:
    if '(' in expr and ')' in expr:
        inside = expr[expr.find('(')+1 : expr.rfind(')')]
        for var in (v.strip() for v in inside.split(',') if v.strip()):
            if var not in seen:
                seen.add(var)
                vars_list.append(var)

print(vars_list)

['X1', 'X2', 'X3']


In [55]:
import random
rand_num = random.randint(1,3)
rand_num

1

In [ ]:
# create combinations (X1, X2, X3) where X1 = 1..n, X2 = X1+1, X3 = X2+1
# if an integer `n` exists in globals() and is > 0 it will be used; otherwise default to 5
n_val = 100
possible_combinations = []
for i in range(1, n_val + 1):
    rand_num = random.randint(1,3)
    
    if rand_num == 1:
        possible_combinations.append((i-1, i, i + 1))
    elif rand_num == 2:
        possible_combinations.append((i-1, i + 1, i + 2))
    else:
        possible_combinations.append((i, i + 1, i + 2))
# possible_combinations = [(x-1, x, x + 1) for x in range(1, n_val + 1)]

print(f"n = {n_val}, total = {len(possible_combinations)}")
print(possible_combinations)

n = 100, total = 100
[(0, 1, 2), (1, 2, 3), (2, 3, 4), (3, 4, 5), (4, 5, 6), (5, 6, 7), (6, 7, 8), (7, 8, 9), (8, 9, 10), (9, 10, 11), (10, 11, 12), (11, 12, 13), (12, 13, 14), (13, 14, 15), (14, 15, 16), (15, 16, 17), (16, 17, 18), (17, 18, 19), (18, 19, 20), (19, 20, 21), (20, 21, 22), (21, 22, 23), (22, 23, 24), (23, 24, 25), (24, 25, 26), (25, 26, 27), (26, 27, 28), (27, 28, 29), (28, 29, 30), (29, 30, 31), (30, 31, 32), (31, 32, 33), (32, 33, 34), (33, 34, 35), (34, 35, 36), (35, 36, 37), (36, 37, 38), (37, 38, 39), (38, 39, 40), (39, 40, 41), (40, 41, 42), (41, 42, 43), (42, 43, 44), (43, 44, 45), (44, 45, 46), (45, 46, 47), (46, 47, 48), (47, 48, 49), (48, 49, 50), (49, 50, 51), (50, 51, 52), (51, 52, 53), (52, 53, 54), (53, 54, 55), (54, 55, 56), (55, 56, 57), (56, 57, 58), (57, 58, 59), (58, 59, 60), (59, 60, 61), (60, 61, 62), (61, 62, 63), (62, 63, 64), (63, 64, 65), (64, 65, 66), (65, 66, 67), (66, 67, 68), (67, 68, 69), (68, 69, 70), (69, 70, 71), (70, 71, 72), (71, 72, 73

In [38]:
from problog.program import PrologString
from problog import get_evaluatable

# Evaluate each combination using ProbLog
results = []
# print("Evaluating combinations:")

all_pred_valuations = []

for combo in possible_combinations:
    # Create mapping: X1 -> combo[0], X2 -> combo[1], X3 -> combo[2]
    mapping = {}
    for i, var in enumerate(['X1', 'X2', 'X3']):
        mapping[var] = combo[i]
    # print(f"Mapping: {mapping}")
    # Create query string
    pot_body_pred_list = []
    for pred in all_preds:
        for var, val in mapping.items():
            pred = pred.replace(var, str(val))
        pot_body_pred_list.append(pred)
    # print(f"Body predicates: {pot_body_pred_list}")

    # construct the queries
    query = """"""
    for str_pred in pot_body_pred_list:
        query = query + f"""
        query({str_pred}).
        """
    # PrologString cannot be concatenated with a plain str, so create a new PrologString
    # from the combined source (dataset is already a str)
    Input = PrologString(dataset + query)
    result = get_evaluatable().create_from(Input).evaluate()
    # print(f"Query: {str_pred} => Result: {result}")
    results.append((mapping, str_pred, result))

    # extract the valuations for all predicates to a list
    pred_values = list(result.values())

    all_pred_valuations.append(pred_values)

body_values = [row[:-1] for row in all_pred_valuations]
head_values = [row[-1] for row in all_pred_valuations]

# print("Body values:")
# print(body_values)
# print("Head values:")
# print(head_values)

# create trainable pytorch tensors
import torch

body_tensor = torch.tensor(body_values, dtype=torch.float32)
head_tensor = torch.tensor(head_values, dtype=torch.float32)

print("Body tensor:")
print(body_tensor)
print("Head tensor:")
print(head_tensor)



Body tensor:
tensor([[1., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],
        [0., 0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0., 1.],

In [39]:
# aggregate entities per variable across all predicates
entities_by_var = {v: set() for v in vars_list}

for pred, arg_map in pred_arity_entity_dict.items():
    for var, vals in arg_map.items():
        if var not in entities_by_var:
            entities_by_var[var] = set()
        for val in vals:
            entities_by_var[var].add(val)

# convert to sorted lists for reproducible output
entities_by_var = {var: sorted(vals) for var, vals in entities_by_var.items()}

# display results
for var in sorted(entities_by_var.keys(), key=lambda x: int(x[1:])):
    print(f"{var}: {entities_by_var[var]} (count={len(entities_by_var[var])})")

X1: ['e11', 'e12', 'e13', 'e14', 'e15', 'e16', 'e17', 'e18', 'e19'] (count=9)
X2: ['e21', 'e22', 'e23', 'e24', 'e27', 'e28', 'e29'] (count=7)
X3: ['e31', 'e32', 'e33', 'e336', 'e38', 'e39'] (count=6)
X4: ['e41', 'e414', 'e42', 'e43', 'e49'] (count=5)
X5: ['e51', 'e52'] (count=2)


In [40]:
import itertools

# build or reuse aggregated entities per variable
if 'entities_by_var' not in globals():
    entities_by_var = {v: set() for v in vars_list}
    for pred, arg_map in pred_arity_entity_dict.items():
        for var, vals in arg_map.items():
            entities_by_var.setdefault(var, set()).update(vals)
    entities_by_var = {var: sorted(vals) for var, vals in entities_by_var.items()}

# prepare lists in X1..X5 order
lists = [entities_by_var.get(var, []) for var in vars_list]

# create all combinations (tuples of values for X1..X5)
variable_entity_combinations = [tuple(items) for items in itertools.product(*lists)]

# optional: inspect
print(f"Total combinations: {len(variable_entity_combinations)}")
print("First 10 combinations:", variable_entity_combinations[:10])

Total combinations: 3780
First 10 combinations: [('e11', 'e21', 'e31', 'e41', 'e51'), ('e11', 'e21', 'e31', 'e41', 'e52'), ('e11', 'e21', 'e31', 'e414', 'e51'), ('e11', 'e21', 'e31', 'e414', 'e52'), ('e11', 'e21', 'e31', 'e42', 'e51'), ('e11', 'e21', 'e31', 'e42', 'e52'), ('e11', 'e21', 'e31', 'e43', 'e51'), ('e11', 'e21', 'e31', 'e43', 'e52'), ('e11', 'e21', 'e31', 'e49', 'e51'), ('e11', 'e21', 'e31', 'e49', 'e52')]


In [41]:
# evaluate each potential body predicate for every combination
# result: valuations_matrix is a list of rows (one per combination),
# each row is a list of 0/1 valuations aligned with `potential_body_preds`

# parse potential_body_preds into (predname, [vars])
parsed_body = []
for expr in potential_body_preds:
    if '(' in expr and ')' in expr:
        predname = expr[:expr.find('(')].strip()
        inside = expr[expr.find('(')+1:expr.rfind(')')]
        arg_vars = [v.strip() for v in inside.split(',') if v.strip()]
    else:
        predname = expr.strip()
        arg_vars = []
    parsed_body.append((predname, arg_vars))

valuations_matrix = []
for combo in variable_entity_combinations:
    mapping = dict(zip(vars_list, combo))
    row = []
    for predname, arg_vars in parsed_body:
        arg_literals = [repr(mapping[var]) for var in arg_vars]  # quoted/escaped literals
        # build regex that matches the predicate instance in the dataset (allows whitespace)
        pat = r'\b' + re.escape(predname) + r'\s*\(\s*' + r'\s*,\s*'.join(map(re.escape, arg_literals)) + r'\s*\)\s*\.'
        found = re.search(pat, dataset) is not None
        row.append(1 if found else 0)
    valuations_matrix.append(row)

# quick inspection
print(f"Total combinations evaluated: {len(valuations_matrix)}")
print(f"Number of body predicates checked per combination: {len(potential_body_preds)}")
print("First 10 valuation rows:", valuations_matrix[:10])

Total combinations evaluated: 3780
Number of body predicates checked per combination: 7
First 10 valuation rows: [[1, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1]]


In [42]:
valuations_matrix

[[1, 0, 0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [1, 1, 0, 0, 0, 0, 1],
 [1, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0,

In [43]:
# deduplicate valuations_matrix (keep first occurrence) and align variable_entity_combinations
seen = set()
unique_valuations = []
unique_combinations = []

for combo, row in zip(variable_entity_combinations, valuations_matrix):
    key = tuple(row)
    if key not in seen:
        seen.add(key)
        unique_combinations.append(combo)
        unique_valuations.append(row)

# replace originals with deduplicated versions
variable_entity_combinations = unique_combinations
valuations_matrix = unique_valuations

print(f"Kept {len(valuations_matrix)} unique valuation rows out of {len(seen) + 0} evaluated.")
print("Unique valuations:", valuations_matrix)
print("Corresponding combinations:", variable_entity_combinations)

Kept 14 unique valuation rows out of 14 evaluated.
Unique valuations: [[1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 1, 1], [1, 1, 0, 0, 0, 0, 1], [0, 1, 0, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 1], [0, 0, 1, 1, 0, 0, 1], [0, 0, 0, 1, 0, 1, 1], [0, 0, 0, 0, 1, 0, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0]]
Corresponding combinations: [('e11', 'e21', 'e31', 'e41', 'e51'), ('e11', 'e21', 'e31', 'e414', 'e51'), ('e11', 'e21', 'e31', 'e414', 'e52'), ('e11', 'e21', 'e32', 'e41', 'e51'), ('e11', 'e21', 'e32', 'e414', 'e51'), ('e12', 'e22', 'e31', 'e41', 'e51'), ('e12', 'e22', 'e32', 'e41', 'e51'), ('e12', 'e24', 'e31', 'e42', 'e51'), ('e13', 'e21', 'e31', 'e41', 'e51'), ('e14', 'e21', 'e31', 'e41', 'e51'), ('e14', 'e22', 'e32', 'e41', 'e51'), ('e14', 'e23', 'e31', 'e41', 'e51'), ('e15', 'e21', 'e31', 'e41', 'e51'), ('e17', 'e28', 'e39', 'e41', 'e51')]
